# Agribusiness Performance Analysis Using SQL

**Week 3 Task | Google Colab Notebook**

This notebook uses SQLite through Python to analyze a hypothetical agribusiness. It expects two CSV files: `farms.csv` and `production.csv`. The data is synthetic and for learning purposes only.

**How to use:** Run the cells from top to bottom. When prompted, upload both CSV files supplied with the assignment.

In [ ]:
# Import the libraries used in this analysis
import sqlite3
import pandas as pd
from google.colab import files
from IPython.display import display

print('Libraries imported successfully.')

## 1. Upload the dataset files

Upload `farms.csv` and `production.csv` when the file picker opens.

In [ ]:
uploaded = files.upload()
print('Uploaded files:', list(uploaded.keys()))

farms = pd.read_csv('farms.csv')
production = pd.read_csv('production.csv')

print('Farm records:', len(farms))
print('Production records:', len(production))
display(farms.head())
display(production.head())

## 2. Create SQLite database and load tables

The `farms` table stores farm names. The `production` table stores one record per farm, crop and season.

In [ ]:
conn = sqlite3.connect(':memory:')

farms.to_sql('farms', conn, if_exists='replace', index=False)
production.to_sql('production', conn, if_exists='replace', index=False)

print('SQLite tables created and CSV data loaded.')
display(pd.read_sql_query('SELECT * FROM farms', conn))
display(pd.read_sql_query('SELECT * FROM production LIMIT 5', conn))

## 3. Data quality checks

**Purpose:** Find records with invalid area, production or operating costs before calculating KPIs.

In [ ]:
query = '''
SELECT * FROM production
WHERE area_acres <= 0 OR production_tonnes < 0
   OR seed_cost < 0 OR fertilizer_cost < 0 OR labor_cost < 0
   OR irrigation_cost < 0 OR other_cost < 0;
'''
display(pd.read_sql_query(query, conn))

## 4. Crop performance KPIs

**Purpose:** Calculate weighted yield per acre, operating cost per kg, revenue, operating profit and profit margin for each crop. Weighted ratios use totals rather than averaging farm-level ratios.

In [ ]:
query = '''
WITH base AS (
  SELECT crop,
         SUM(area_acres) AS area_acres,
         SUM(production_tonnes) AS production_tonnes,
         SUM(seed_cost + fertilizer_cost + labor_cost + irrigation_cost + other_cost) AS total_cost,
         SUM(production_tonnes * 1000.0 * selling_price_per_kg) AS revenue
  FROM production
  GROUP BY crop
)
SELECT crop, area_acres, production_tonnes,
       ROUND(production_tonnes / NULLIF(area_acres, 0), 2) AS yield_tonnes_per_acre,
       ROUND(total_cost, 2) AS total_cost,
       ROUND(total_cost / NULLIF(production_tonnes * 1000.0, 0), 2) AS cost_per_kg,
       ROUND(revenue, 2) AS revenue,
       ROUND(revenue - total_cost, 2) AS operating_profit,
       ROUND((revenue - total_cost) / NULLIF(revenue, 0), 4) AS profit_margin
FROM base
ORDER BY operating_profit DESC;
'''
crop_kpis = pd.read_sql_query(query, conn)
display(crop_kpis)

## 5. Seasonal performance trend

**Purpose:** Compare weighted yield and cost per kg across agricultural seasons to identify weak or strong periods.

In [ ]:
query = '''
SELECT season,
       SUM(area_acres) AS area_acres,
       SUM(production_tonnes) AS production_tonnes,
       ROUND(SUM(production_tonnes) / NULLIF(SUM(area_acres), 0), 2) AS yield_tonnes_per_acre,
       ROUND(SUM(seed_cost + fertilizer_cost + labor_cost + irrigation_cost + other_cost)
             / NULLIF(SUM(production_tonnes) * 1000.0, 0), 2) AS cost_per_kg
FROM production
GROUP BY season
ORDER BY season;
'''
season_kpis = pd.read_sql_query(query, conn)
display(season_kpis)

## 6. Farm-level efficiency

**Purpose:** Compare yield and cost per kg for each farm and crop. This helps identify operational differences within the same crop.

In [ ]:
query = '''
SELECT f.farm_name, p.crop,
       ROUND(SUM(p.production_tonnes) / NULLIF(SUM(p.area_acres), 0), 2) AS yield_tonnes_per_acre,
       ROUND(SUM(p.seed_cost + p.fertilizer_cost + p.labor_cost + p.irrigation_cost + p.other_cost)
             / NULLIF(SUM(p.production_tonnes) * 1000.0, 0), 2) AS cost_per_kg
FROM production p
JOIN farms f ON f.farm_id = p.farm_id
GROUP BY f.farm_name, p.crop
ORDER BY p.crop, yield_tonnes_per_acre DESC;
'''
farm_efficiency = pd.read_sql_query(query, conn)
display(farm_efficiency)

## 7. Operating cost composition

**Purpose:** Identify which input category accounts for the largest share of operating costs.

In [ ]:
query = '''
SELECT 'Seed' AS cost_category, SUM(seed_cost) AS amount FROM production
UNION ALL SELECT 'Fertilizer', SUM(fertilizer_cost) FROM production
UNION ALL SELECT 'Labor', SUM(labor_cost) FROM production
UNION ALL SELECT 'Irrigation', SUM(irrigation_cost) FROM production
UNION ALL SELECT 'Other', SUM(other_cost) FROM production
ORDER BY amount DESC;
'''
cost_composition = pd.read_sql_query(query, conn)
cost_composition['share_percent'] = 100 * cost_composition['amount'] / cost_composition['amount'].sum()
display(cost_composition)

## 8. Profitability by crop and season

**Purpose:** Find the crop-season combinations that generate the highest operating profit.

In [ ]:
query = '''
SELECT crop, season,
       SUM(production_tonnes * 1000.0 * selling_price_per_kg) AS revenue,
       SUM(seed_cost + fertilizer_cost + labor_cost + irrigation_cost + other_cost) AS total_cost,
       SUM(production_tonnes * 1000.0 * selling_price_per_kg)
         - SUM(seed_cost + fertilizer_cost + labor_cost + irrigation_cost + other_cost) AS operating_profit
FROM production
GROUP BY crop, season
ORDER BY operating_profit DESC;
'''
display(pd.read_sql_query(query, conn))

## 9. Rank seasons by yield within each crop

**Purpose:** Use a SQL window function to rank the seasons separately for rice, maize and potato.

In [ ]:
query = '''
SELECT crop, season, yield_tonnes_per_acre,
       RANK() OVER (PARTITION BY crop ORDER BY yield_tonnes_per_acre DESC) AS yield_rank
FROM (
  SELECT crop, season,
         SUM(production_tonnes) / NULLIF(SUM(area_acres), 0) AS yield_tonnes_per_acre
  FROM production
  GROUP BY crop, season
);
'''
display(pd.read_sql_query(query, conn))

## 10. Overall business dashboard

**Purpose:** Produce the headline KPIs for the full agribusiness.

In [ ]:
query = '''
SELECT
  SUM(area_acres) AS total_area_acres,
  SUM(production_tonnes) AS total_production_tonnes,
  SUM(production_tonnes) / NULLIF(SUM(area_acres), 0) AS overall_yield_tonnes_per_acre,
  SUM(seed_cost + fertilizer_cost + labor_cost + irrigation_cost + other_cost)
    / NULLIF(SUM(production_tonnes) * 1000.0, 0) AS overall_cost_per_kg,
  SUM(production_tonnes * 1000.0 * selling_price_per_kg) AS revenue,
  SUM(production_tonnes * 1000.0 * selling_price_per_kg)
    - SUM(seed_cost + fertilizer_cost + labor_cost + irrigation_cost + other_cost) AS operating_profit
FROM production;
'''
dashboard = pd.read_sql_query(query, conn)
display(dashboard)

## 11. Visualize key results

The charts below are generated from the SQL query results.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
plt.bar(crop_kpis['crop'], crop_kpis['yield_tonnes_per_acre'])
plt.title('Weighted yield by crop')
plt.xlabel('Crop')
plt.ylabel('Tonnes per acre')
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(season_kpis['season'], season_kpis['yield_tonnes_per_acre'], marker='o')
plt.title('Seasonal weighted yield trend')
plt.xlabel('Season')
plt.ylabel('Tonnes per acre')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.bar(cost_composition['cost_category'], cost_composition['amount'])
plt.title('Operating cost composition')
plt.xlabel('Cost category')
plt.ylabel('INR')
plt.tight_layout()
plt.show()

## 12. Export the analysis outputs

Save the main KPI result tables as CSV files and download them from Colab.

In [ ]:
crop_kpis.to_csv('crop_kpis_output.csv', index=False)
season_kpis.to_csv('season_kpis_output.csv', index=False)
farm_efficiency.to_csv('farm_efficiency_output.csv', index=False)
cost_composition.to_csv('cost_composition_output.csv', index=False)
print('Output CSV files created.')
files.download('crop_kpis_output.csv')
files.download('season_kpis_output.csv')
files.download('farm_efficiency_output.csv')
files.download('cost_composition_output.csv')

## Conclusion

This notebook demonstrates how SQL can be used to evaluate agribusiness performance through yield, unit cost, revenue, operating profit, seasonal trends and farm-level efficiency. Interpret the results as illustrative because the dataset is simulated. Replace the sample CSV files with verified business data before making real operational decisions.